In [ ]:
from nocode_robot_programming.state_decision.utils import kill_other_ipykernels
kill_other_ipykernels(force=True)
import trajectory_data
import matplotlib.pyplot as plt
from nocode_robot_programming.state_decision.dataloader import TrajectoryDataset
from nocode_robot_programming.state_decision.dino_model import DINOFeaturePresence, DINOFeaturePresenceConcat, DINOFeaturePresenceAttnGated
from nocode_robot_programming.state_decision.dino_with_mil import DINOWithMIL
from nocode_robot_programming.state_decision.SIFT_model import StateDeciderSIFT
from nocode_robot_programming.state_decision.AEGP_model import AEGP
from nocode_robot_programming.state_decision.state_decider import StateDeciderBase
from nocode_robot_programming.state_decision.utils import Filename
from gesture_detector.utils import pretty_confusion_matrix
import torch
import numpy as np

from trajectory_data.skill_visualizer import play_video
from nocode_robot_programming.state_decision.utils import add_tag, likelihood_sparklines, likelihood_sparklines_withtruelabels, likelihood_single_axis
from nocode_robot_programming.state_decision.utils import visualize_accuracies
from nocode_robot_programming.state_decision.dataloader import ImageDatasetView, saved_img_processing
from nocode_robot_programming.jupyter_plot import jupyter_plot as ipt, show_gray_video_cuda

seed = 48
np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

# from nocode_robot_programming.state_decision.dataset_D1 import d1_peg_pick, d2_peg_pick, d1_probe, d2_probe, dupl, get_dataset_view
# from nocode_robot_programming.state_decision.dataset_D2 import d1_move
from nocode_robot_programming.state_decision.dataset_D2 import d1_anomaly_peg_pick, d2_anomaly_peg_pick, d3_anomaly_peg_pick, d1_anomaly_probe, d2_anomaly_probe, d3_anomaly_probe
datafileloader = TrajectoryDataset(trajectory_data.package_path)

You can see dataset includes these tasks, for each tasks there are several trials.

# List through the images `X` from the dataset and its labels `y`

- Tasks as load functions: `datasets = d1_peg_pick, d2_peg_pick, d1_probe, d2_probe`
- Each dataset has difficulty mixes: `hard_mix, medium_mix, easy_mix = datasets[i]`
- Each mix has train, test, and description `d_train, d_test, d_text = <easy|medium|difficult>_mix`
- `d_train` and `d_test` are 

In [ ]:
datasets = d1_spawned_box_2cls(datafileloader)
d_train, d_test, d_text = datasets[0] # hard mix
print(f"{d_text=},\n{d_train.X.shape=},\n{d_train.y_int.shape=},\n{len(d_train.y_names)=},\n{d_train.y_cls=}")

In [ ]:
d_train.showcase_captions(fps=20, scale=2.0)
d_test.showcase_captions(fps=20, scale=2.0)

In [ ]:
d_train.play_video(fps=5, scale=2.0)
d_test.play_video(fps=10, scale=2.0)

# Checkpoint 2025-10-29

- Test accuracy achieved 93%

In [ ]:
train_stats, test_stats, model_names = [], [], []
for decider in [
        DINOFeaturePresence(dino_variant= "dinov2_vits14", percentile_keep=0.1),
        DINOFeaturePresence(dino_variant= "dinov2_vits14", percentile_keep=0.2),
        DINOFeaturePresence(dino_variant= "dinov2_vits14", percentile_keep=0.3),

        DINOFeaturePresenceConcat(dino_variant= "dinov2_vits14", percentile_keep=0.1),
        DINOFeaturePresenceConcat(dino_variant= "dinov2_vits14", percentile_keep=0.2),
        DINOFeaturePresenceConcat(dino_variant= "dinov2_vits14", percentile_keep=0.3),

        DINOFeaturePresenceAttnGated(dino_variant= "dinov2_vits14", percentile_keep=0.1, attn_mode = "hard", head_reduce = "mean", attn_keep = 0.4),
        DINOFeaturePresenceAttnGated(dino_variant= "dinov2_vits14", percentile_keep=0.2, attn_mode = "hard", head_reduce = "mean", attn_keep = 0.4),
        DINOFeaturePresenceAttnGated(dino_variant= "dinov2_vits14", percentile_keep=0.3, attn_mode = "hard", head_reduce = "mean", attn_keep = 0.4),

        DINOFeaturePresenceAttnGated(dino_variant= "dinov2_vits14", percentile_keep=0.1, attn_mode = "hard", head_reduce = "max" , attn_keep = 0.4),
        DINOFeaturePresenceAttnGated(dino_variant= "dinov2_vits14", percentile_keep=0.2, attn_mode = "hard", head_reduce = "max" , attn_keep = 0.4),
        DINOFeaturePresenceAttnGated(dino_variant= "dinov2_vits14", percentile_keep=0.3, attn_mode = "hard", head_reduce = "max" , attn_keep = 0.4),
        
        DINOWithMIL(dino_variant= "dinov2_vits14", percentile_keep=0.1, att_hidden = 128, dropout_p = 0.1, lr = 1e-4, weight_decay = 1e-3, epochs = 1000),
        DINOWithMIL(dino_variant= "dinov2_vits14", percentile_keep=0.2, att_hidden = 128, dropout_p = 0.1, lr = 1e-4, weight_decay = 1e-3, epochs = 1000),
        DINOWithMIL(dino_variant= "dinov2_vits14", percentile_keep=0.3, att_hidden = 128, dropout_p = 0.1, lr = 1e-4, weight_decay = 1e-3, epochs = 1000),
        
        # DINOFeaturePresence(dino_variant="dinov2_vitg14", percentile_keep=0.1),
        # DINOFeaturePresence(dino_variant="dinov2_vitg14", percentile_keep=0.2),
        # DINOFeaturePresence(dino_variant="dinov2_vitg14", percentile_keep=0.3),

        # DINOFeaturePresence(dino_variant = "dinov2_vitl14", percentile_keep=0.1),
        # DINOFeaturePresence(dino_variant = "dinov2_vitl14", percentile_keep=0.2),
        # DINOFeaturePresence(dino_variant = "dinov2_vitl14", percentile_keep=0.3),

        StateDeciderSIFT(method = "SIFT"), # suggesting the background problem
        # # StateDeciderSIFT(method = "AKAZE"),
        StateDeciderSIFT(method = "ORB"),
        
        # AEGP(binary=False, pix=64),
        # AEGP(binary=True, pix=64),
        # AEGP(binary=True, pix=64),
        # AEGP(binary=True, pix=224),

        # AEGP(binary=False, pix=64, crop=False),
        # AEGP(binary=True, pix=64, crop=False),
    ]:
    train_stats_model, test_stats_model = [], []
    dataset_names = []
    for task_dataset_gen in [d1_anomaly_peg_pick, d2_anomaly_peg_pick, d3_anomaly_peg_pick, d1_anomaly_probe, d2_anomaly_probe, d3_anomaly_probe]:
        
        hard_mid_easy = task_dataset_gen(datafileloader)

        for d_train, d_test, d_text in hard_mid_easy:
            print(f"Model: {decider.__class__.__name__}, Dataset: {d_text}")
            
            decider.train(d_train.X, d_train.y_int, d_train.y_cls); ipt.save() # loss fig obtained in train function

            y_pred = decider.predict_many(d_train.X)
            pretty_confusion_matrix.pp_matrix_from_string_data(d_train.y_names, y_pred, name=f"d_train,{decider}", show=False); ipt.save()
            train_stats_model.append((np.array(d_train.y_names) == np.array(y_pred)).mean())

            y_pred = decider.predict_many(d_test.X)
            pretty_confusion_matrix.pp_matrix_from_string_data(d_test.y_names, y_pred, name=f"d_test,{decider}", show=False); ipt.save()
            test_stats_model.append((np.array(d_test.y_names) == np.array(y_pred)).mean())
            
            # ipt.show(small=True)
            ipt.delete() # don't plot
            dataset_names.append(d_text.split(",")[0])

    train_stats.append(train_stats_model)
    test_stats.append(test_stats_model)
    model_names.append(decider.__str__())

In [ ]:
visualize_accuracies(100 * np.array(train_stats), 100 * np.array(test_stats), model_names, dataset_names, out_dir="plot", jupyter_plot=True)
f"Overall accuracy on train data: {round(100 * np.array(train_stats).mean())}% and test data {round(100 * np.array(test_stats).mean())}%"

In [ ]:
pretty_confusion_matrix.pp_matrix_from_string_data(d_test.y_names, y_pred, name=f"d_test,{decider}", show=False); ipt.save()
ipt.show()
print((np.array(d_test.y_names) == y_pred).mean())
